In [1]:
import json
import os
import random
import time
import sqlite3
from pathlib import Path
from datetime import date, timedelta
from typing import Dict, Iterable, List, Optional, Tuple

import requests

In [3]:
import requests

API_KEY = "9053ff50-b2c9-40e7-b9ce-def1d57d5dc0"
r = requests.get(
    "https://content.guardianapis.com/sections",
    params={"api-key": API_KEY, "format": "json", "page-size": 200},
    timeout=30
)
r.raise_for_status()
data = r.json()["response"]["results"]

for s in data:
    print(f"{s['id']:25s}  |  {s['webTitle']}")

about                      |  About
animals-farmed             |  Animals farmed
artanddesign               |  Art and design
australia-news             |  Australia news
better-business            |  Better Business
books                      |  Books
business                   |  Business
business-to-business       |  Business to business
cardiff                    |  Cardiff
childrens-books-site       |  Children's books
cities                     |  Cities
commentisfree              |  Opinion
community                  |  Community
crosswords                 |  Crosswords
culture                    |  Culture
culture-network            |  Culture Network
culture-professionals-network  |  Culture professionals network
edinburgh                  |  Edinburgh
education                  |  Education
enterprise-network         |  Guardian Enterprise Network
environment                |  Environment
extra                      |  Extra
fashion                    |  Fashion
film          

In [3]:
# USER CONFIG
# -------------------------

API_KEY = os.getenv("GUARDIAN_API_KEY", "").strip()
if not API_KEY:
    # You can keep it here if you prefer (less secure than env var)
    API_KEY = ""

ARCHIVE_DIR = Path("/Users/faridaishakova/Library/CloudStorage/OneDrive-NewcastleUniversity/The Guardian_full")

# Keep only the requested Guardian sections (API section IDs)
SECTIONS = [
    "world",                          # World news
    "business",                       # Business
    "uk-news",                        # UK news
    "politics",                       # Politics
    "society",                        # Society
    "money",                          # Money
    "education",                      # Education
    "environment",                    # Environment
    "technology",                     # Technology
    "news",                           # News
    "food",                           # Food
    "global-development",             # Global development
    "travel",                         # Travel
    "lifeandstyle",                   # Life and style
    "healthcare-network",             # Healthcare Professionals Network
    "science",                        # Science
    "housing-network",                # Housing Network
    "inequality",                     # Inequality
    "business-to-business",           # Business to business
    "cities",                         # Cities
]

In [5]:
# Period (edit as needed)
START_YEAR = 2008
END_YEAR   = 2025

In [7]:
# If you want absolutely no query restriction, keep None (recommended for now since you narrowed sections)
QUERY: Optional[str] = None

In [9]:
# Pull full text
INCLUDE_TEXT = True  # -> show-fields includes bodyText

In [11]:
# API paging / politeness
PAGE_SIZE = 200               # max is typically 200
SLEEP_SECONDS = 1.6           # base sleep between calls (be conservative)
MAX_CALLS_PER_RUN = 450       # soft budget per run (optional); set very large if you want continuous runs

In [13]:
# Rate limit behavior
AUTO_WAIT_ON_429 = True       # if True, script sleeps Retry-After and continues
LONG_RETRY_AFTER_SECONDS = 3600  # treat >1h as "quota-like" wait
MAX_SLEEP_ON_429 = 6 * 3600      # cap single wait to 6h (safety). If Retry-After > cap, it will sleep cap and retry.

In [15]:
API_ENDPOINT_SEARCH = "https://content.guardianapis.com/search"

In [17]:
# HELPERS: dates, state
# -------------------------
def month_ranges(d0: date, d1: date) -> Iterable[Tuple[date, date]]:
    """Yield (month_start, month_end) covering d0..d1 inclusive."""
    cur = date(d0.year, d0.month, 1)
    while cur <= d1:
        nxt = date(cur.year + (cur.month // 12), (cur.month % 12) + 1, 1)
        month_end = min(d1, nxt - timedelta(days=1))
        yield cur, month_end
        cur = nxt

In [19]:
def load_state(path: Path) -> Dict:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}

In [21]:
def save_state(path: Path, state: Dict) -> None:
    path.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")

In [23]:
def run_signature(year: int) -> Dict:
    """Used to detect config changes and avoid reusing incompatible state."""
    return {
        "year": year,
        "sections": SECTIONS,
        "query": QUERY,
        "include_text": INCLUDE_TEXT,
        "page_size": PAGE_SIZE,
        "journal_mode": "DELETE",
    }

In [25]:
# DB schema + upsert
# -------------------------
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS articles (
    id TEXT PRIMARY KEY,
    section_id TEXT,
    section_name TEXT,
    requested_section TEXT,
    type TEXT,
    web_publication_date TEXT,
    web_title TEXT,
    web_url TEXT,
    headline TEXT,
    byline TEXT,
    body_text TEXT,
    fetched_at TEXT
);
"""

In [27]:
def init_db(db_path: Path) -> None:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(db_path) as conn:
        # For OneDrive/Cloud folders it's often safer to avoid WAL side-files
        conn.execute("PRAGMA journal_mode=DELETE;")
        conn.execute("PRAGMA synchronous=NORMAL;")
        conn.execute(CREATE_TABLE_SQL)

        # Add missing columns if DB already exists with older schema
        ensure_column(conn, "articles", "requested_section", "TEXT")
        ensure_column(conn, "articles", "body_text", "TEXT")
        ensure_column(conn, "articles", "headline", "TEXT")
        ensure_column(conn, "articles", "byline", "TEXT")

        conn.execute("CREATE INDEX IF NOT EXISTS idx_articles_pubdate ON articles(web_publication_date);")
        conn.execute("CREATE INDEX IF NOT EXISTS idx_articles_section ON articles(section_id);")
        conn.execute("CREATE INDEX IF NOT EXISTS idx_articles_req_section ON articles(requested_section);")
        conn.commit()

In [29]:
def ensure_column(conn: sqlite3.Connection, table: str, col: str, coltype: str) -> None:
    cols = [r[1] for r in conn.execute(f"PRAGMA table_info({table});").fetchall()]
    if col not in cols:
        conn.execute(f"ALTER TABLE {table} ADD COLUMN {col} {coltype};")

In [31]:
def upsert_many(conn: sqlite3.Connection, rows: List[Dict]) -> int:
    if not rows:
        return 0
    conn.executemany(
        """
        INSERT INTO articles (
            id, section_id, section_name, requested_section, type,
            web_publication_date, web_title, web_url,
            headline, byline, body_text, fetched_at
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, datetime('now'))
        ON CONFLICT(id) DO UPDATE SET
            section_id=excluded.section_id,
            section_name=excluded.section_name,
            requested_section=excluded.requested_section,
            type=excluded.type,
            web_publication_date=excluded.web_publication_date,
            web_title=excluded.web_title,
            web_url=excluded.web_url,
            headline=excluded.headline,
            byline=excluded.byline,
            body_text=excluded.body_text,
            fetched_at=datetime('now');
        """,
        [
            (
                r.get("id"),
                r.get("section_id"),
                r.get("section_name"),
                r.get("requested_section"),
                r.get("type"),
                r.get("web_publication_date"),
                r.get("web_title"),
                r.get("web_url"),
                r.get("headline"),
                r.get("byline"),
                r.get("body_text"),
            )
            for r in rows
        ],
    )
    return len(rows)

In [33]:
# Guardian API call with retry/backoff
# -------------------------
def guardian_search(
    session: requests.Session,
    api_key: str,
    section: str,
    from_date: date,
    to_date: date,
    page: int,
    page_size: int,
    include_text: bool,
    query: Optional[str],
) -> Dict:
    params = {
        "api-key": api_key,
        "section": section,
        "from-date": from_date.isoformat(),
        "to-date": to_date.isoformat(),
        "order-by": "oldest",
        "page": page,
        "page-size": page_size,
    }
    if query:
        params["q"] = query

    fields = ["headline", "byline"]
    if include_text:
        fields.append("bodyText")
    params["show-fields"] = ",".join(fields)

    # retry loop
    for attempt in range(1, 7):
        try:
            r = session.get(API_ENDPOINT_SEARCH, params=params, timeout=45)
        except requests.RequestException as e:
            wait = min(60, 2 ** attempt)
            print(f"Network error: {e} | attempt {attempt} -> sleep {wait}s")
            time.sleep(wait)
            continue

        if r.status_code == 429:
            retry_after = r.headers.get("Retry-After")
            ra = int(retry_after) if (retry_after and retry_after.isdigit()) else None
            print(f"HTTP 429 | Retry-After={ra} | attempt {attempt}")

            if not AUTO_WAIT_ON_429:
                # Return special payload; caller may save state and exit
                return {"__rate_limited__": True, "__retry_after__": ra}

            # Sleep and retry
            if ra is None:
                wait = min(180, 2 ** attempt * 10)
            else:
                wait = min(ra + 5, MAX_SLEEP_ON_429)

            # If it's a long wait (quota-like), still we can wait and continue automatically
            if ra is not None and ra >= LONG_RETRY_AFTER_SECONDS:
                print(f"Rate limit/quota-like 429. Sleeping {wait}s (~{wait/3600:.2f}h) then resuming...")

            time.sleep(wait)
            continue

        if 500 <= r.status_code < 600:
            wait = min(180, 2 ** attempt * 5)
            print(f"HTTP {r.status_code} server error | attempt {attempt} -> sleep {wait}s")
            time.sleep(wait)
            continue

        r.raise_for_status()
        return r.json()

    # if all attempts failed
    raise RuntimeError("Failed after retries (guardian_search).")

In [35]:
def normalize_item(item: Dict, requested_section: str) -> Dict:
    fields = item.get("fields", {}) or {}
    return {
        "id": item.get("id"),
        "type": item.get("type"),
        "section_id": item.get("sectionId"),
        "section_name": item.get("sectionName"),
        "requested_section": requested_section,
        "web_publication_date": item.get("webPublicationDate"),
        "web_title": item.get("webTitle"),
        "web_url": item.get("webUrl"),
        "headline": fields.get("headline"),
        "byline": fields.get("byline"),
        "body_text": fields.get("bodyText"),
    }

In [37]:
# Main runners (year -> all years)
# -------------------------
def year_paths(year: int) -> Tuple[Path, Path]:
    db_path = ARCHIVE_DIR / f"guardian_{year}.sqlite"
    state_path = ARCHIVE_DIR / f"guardian_{year}_state.json"
    return db_path, state_path

In [39]:
def reset_year(year: int) -> None:
    db_path, state_path = year_paths(year)
    if db_path.exists():
        db_path.unlink()
    if state_path.exists():
        state_path.unlink()
    print(f"[{year}] Reset: removed DB + state.")

In [41]:
def print_year_stats(year: int) -> None:
    db_path, _ = year_paths(year)
    if not db_path.exists():
        print(f"[{year}] No DB found:", db_path)
        return

    with sqlite3.connect(db_path) as conn:
        cur = conn.cursor()
        cur.execute("SELECT COUNT(*) FROM articles;")
        n = cur.fetchone()[0]
        cur.execute("SELECT MIN(web_publication_date), MAX(web_publication_date) FROM articles;")
        dr = cur.fetchone()
        cur.execute("""
            SELECT
              SUM(CASE WHEN body_text IS NOT NULL AND LENGTH(body_text) > 0 THEN 1 ELSE 0 END) AS with_text,
              SUM(CASE WHEN body_text IS NULL OR LENGTH(body_text) = 0 THEN 1 ELSE 0 END) AS without_text
            FROM articles;
        """)
        wt = cur.fetchone()

    size_mb = db_path.stat().st_size / (1024 * 1024)
    print(f"\n--- YEAR {year} ---")
    print("Rows:", n)
    print("Date range:", dr)
    print("With/without body_text:", wt)
    print("DB size (MB):", round(size_mb, 2))
    print("DB path:", str(db_path))

In [43]:
def run_year(year: int, reset_before: bool = False) -> None:
    if not API_KEY:
        raise SystemExit("Set GUARDIAN_API_KEY env var or paste API_KEY in the notebook.")

    if reset_before:
        reset_year(year)

    db_path, state_path = year_paths(year)
    init_db(db_path)

    state = load_state(state_path)
    sig = run_signature(year)

    # If signature differs, state is incompatible -> reset state (DB kept, but resume resets)
    if state.get("run_sig") != sig:
        state = {"run_sig": sig, "section_index": 0, "month_start": None, "page": 1}
        save_state(state_path, state)

    section_index = int(state.get("section_index", 0))
    month_start_override = state.get("month_start")  # "YYYY-MM-DD" or None
    page_override = int(state.get("page", 1))

    start_date = date(year, 1, 1)
    end_date = date(year, 12, 31)

    session = requests.Session()
    session.headers.update({"User-Agent": "uk-inflation-research-bot/1.0"})

    calls = 0

    for si in range(section_index, len(SECTIONS)):
        section = SECTIONS[si]
        print(f"\n=== YEAR {year} | SECTION: {section} ({si+1}/{len(SECTIONS)}) ===")

        for (m_start, m_end) in month_ranges(start_date, end_date):

            # resume: skip months earlier than saved month_start
            if month_start_override:
                ms = date.fromisoformat(month_start_override)
                if m_start < ms:
                    continue
                start_page = page_override if m_start == ms else 1
            else:
                start_page = 1

            page = start_page

            while True:
                if calls >= MAX_CALLS_PER_RUN:
                    print(f"[{year}] Reached MAX_CALLS_PER_RUN={MAX_CALLS_PER_RUN}. Saving state and exiting chunk.")
                    save_state(state_path, {
                        "run_sig": sig,
                        "section_index": si,
                        "month_start": m_start.isoformat(),
                        "page": page,
                    })
                    return

                data = guardian_search(
                    session=session,
                    api_key=API_KEY,
                    section=section,
                    from_date=m_start,
                    to_date=m_end,
                    page=page,
                    page_size=PAGE_SIZE,
                    include_text=INCLUDE_TEXT,
                    query=QUERY,
                )
                calls += 1
                time.sleep(SLEEP_SECONDS)

                # If AUTO_WAIT_ON_429=False, guardian_search might return this special structure
                if data.get("__rate_limited__"):
                    ra = data.get("__retry_after__")
                    print(f"[{year}] Rate limited. Retry-After={ra}. Saving state and exiting.")
                    save_state(state_path, {
                        "run_sig": sig,
                        "section_index": si,
                        "month_start": m_start.isoformat(),
                        "page": page,
                    })
                    return

                resp = (data.get("response", {}) or {})
                results = (resp.get("results", []) or [])
                pages = int(resp.get("pages", 1) or 1)

                batch = [normalize_item(it, requested_section=section) for it in results]

                with sqlite3.connect(db_path) as conn:
                    conn.execute("PRAGMA journal_mode=DELETE;")
                    conn.execute("PRAGMA synchronous=NORMAL;")
                    wrote = upsert_many(conn, batch)
                    conn.commit()

                print(f"{section} | {m_start}..{m_end} | page {page}/{pages} | wrote {wrote}")

                # Save state after each successful page
                save_state(state_path, {
                    "run_sig": sig,
                    "section_index": si,
                    "month_start": m_start.isoformat(),
                    "page": (page + 1) if page < pages else 1,
                })

                if page >= pages:
                    break
                page += 1

            # Month finished -> clear overrides for next month
            month_start_override = None
            page_override = 1

   # Finished year
    save_state(state_path, {"run_sig": sig, "section_index": len(SECTIONS), "month_start": end_date.isoformat(), "page": 1})
    print(f"\n[{year}] DONE. DB saved at: {db_path}")

In [45]:
def run_all_years(start_year: int = START_YEAR, end_year: int = END_YEAR, reset_each_year: bool = False) -> None:
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("Archive folder:", ARCHIVE_DIR)
    for y in range(start_year, end_year + 1):
        run_year(y, reset_before=reset_each_year)
        print_year_stats(y)

In [47]:
# Quick utilities (optional)
# -------------------------
def show_latest_text(year: int, limit: int = 3) -> None:
    db_path, _ = year_paths(year)
    with sqlite3.connect(db_path) as conn:
        rows = conn.execute(
            """
            SELECT web_publication_date, section_id, web_title, substr(body_text,1,500) AS preview, web_url
            FROM articles
            ORDER BY web_publication_date DESC
            LIMIT ?;
            """,
            (limit,),
        ).fetchall()

    for r in rows:
        print("\n---")
        print("date:", r[0])
        print("section:", r[1])
        print("title:", r[2])
        print("preview:", r[3])
        print("url:", r[4])

In [55]:
run_all_years(2000, 2025, reset_each_year=False)

Archive folder: /Users/faridaishakova/Library/CloudStorage/OneDrive-NewcastleUniversity/The Guardian_full

=== YEAR 2024 | SECTION: news (10/20) ===
news | 2024-04-01..2024-04-30 | page 1/1 | wrote 52
news | 2024-05-01..2024-05-31 | page 1/1 | wrote 70
news | 2024-06-01..2024-06-30 | page 1/1 | wrote 45
news | 2024-07-01..2024-07-31 | page 1/1 | wrote 53
news | 2024-08-01..2024-08-31 | page 1/1 | wrote 59
news | 2024-09-01..2024-09-30 | page 1/1 | wrote 53
news | 2024-10-01..2024-10-31 | page 1/1 | wrote 58
news | 2024-11-01..2024-11-30 | page 1/1 | wrote 51
news | 2024-12-01..2024-12-31 | page 1/1 | wrote 47

=== YEAR 2024 | SECTION: food (11/20) ===
food | 2024-01-01..2024-01-31 | page 1/1 | wrote 99
food | 2024-02-01..2024-02-29 | page 1/1 | wrote 82
food | 2024-03-01..2024-03-31 | page 1/1 | wrote 95
food | 2024-04-01..2024-04-30 | page 1/1 | wrote 87
food | 2024-05-01..2024-05-31 | page 1/1 | wrote 105
food | 2024-06-01..2024-06-30 | page 1/1 | wrote 86
food | 2024-07-01..2024-07-